In [ ]:
import pandas as pd
from csae_pyutils import gsheet_to_df
from tqdm import tqdm
from apis_core.apis_metainfo.models import Collection, Uri
from apis_core.apis_entities.models import Place
from apis_core.apis_vocabularies.models import PlacePlaceRelation, PlaceType
from apis_core.apis_relations.models import PlacePlace


In [ ]:
col = Collection.objects.get(id=83)
col_kraus = Collection.objects.get_or_create(name="Karl Kraus: Vorleser")[0]
place_place_relation = PlacePlaceRelation.objects.get(name="gehört zu")
col, place_place_relation

In [ ]:
df = gsheet_to_df("1VlKiEjix4zx0ixu5nIap74K6wVaH21qikKLUCpfpb0s")

In [ ]:
df.head(3)

In [ ]:
place_types = {}
for i, row in tqdm(df.iterrows(), total=len(df)):
    domain = row["domain"]
    pmb_uri, _ = Uri.objects.get_or_create(uri=row["URI"], domain=domain)
    entity = pmb_uri.entity
    if entity:
        place = Place.objects.get(id=entity.id)
        pass
    else:
        try:
            place_types[row["typ"]]
        except KeyError:
            place_type = PlaceType.objects.get_or_create(name=row["typ"])[0]
            place_types[row["typ"]] = place_type
        item = {
            "name": row["name"][:254] if pd.notna(row["name"]) else "",
            "kind": place_type
        }
        entity = Place.objects.create(**item)
        place = entity
        pmb_uri.entity = entity
        pmb_uri.save()
    if domain == "ansorge":
        entity.collection.add(col)
    else:
        entity.collection.add(col_kraus)
    try:
        related_place = Place.objects.get(id=row["liegt_in_pmb"])
    except:
        print(row["name"], row["liegt_in_pmb"])
        continue
    PlacePlace.objects.get_or_create(related_placea=place, related_placeb=related_place, relation_type=place_place_relation)
